# Test: GamePredictor

Tests `predictor.py` end-to-end:
1. Instantiation and model detection
2. Output shape and types
3. Probability constraints
4. Confidence thresholds
5. Unknown team handling
6. All 30-team matchups
7. Baseline fallback
8. Prediction consistency

## Setup

In [1]:
import os, sys
import pandas as pd

MODEL_DIR = os.path.abspath('.')
BACKEND_DIR = os.path.dirname(MODEL_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

from model.predictor import GamePredictor, _format_prediction
from model.features import TEAM_NAME_TO_ABB

predictor = GamePredictor()

MODEL_PATH = os.path.join(MODEL_DIR, 'artifacts', 'model.pkl')
DATASET_PATH = os.path.join(MODEL_DIR, 'artifacts', 'dataset.pkl')

print(f'model.pkl exists   : {os.path.exists(MODEL_PATH)}')
print(f'dataset.pkl exists : {os.path.exists(DATASET_PATH)}')
print(f'predictor.is_loaded: {predictor.is_loaded()}')
print(f'model version      : {"v1" if predictor._final_features is not None else "v2" if predictor.is_loaded() else "baseline"}')

model.pkl exists   : True
dataset.pkl exists : True
predictor.is_loaded: True
model version      : v1


## 1. Output shape and required keys

In [2]:
result = predictor.predict('Seattle Mariners', 'Minnesota Twins')
print(result)

REQUIRED_KEYS = {'home_win_prob', 'away_win_prob', 'predicted_winner', 'confidence', 'model_used'}
assert set(result.keys()) == REQUIRED_KEYS, f'Missing keys: {REQUIRED_KEYS - set(result.keys())}'

assert isinstance(result['home_win_prob'], float)
assert isinstance(result['away_win_prob'], float)
assert isinstance(result['predicted_winner'], str)
assert isinstance(result['confidence'], str)
assert isinstance(result['model_used'], str)

print('Output shape: OK')

{'home_win_prob': 0.535, 'away_win_prob': 0.465, 'predicted_winner': 'Seattle Mariners', 'confidence': 'low', 'model_used': 'ML Model v1'}
Output shape: OK


/Users/nickchapman/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [4]:
get_matchup_features??

Signature:
get_matchup_features(
    home_team: str,
    away_team: str,
    final_features: List[str],
    year: Optional[int] = None,
) -> Optional[numpy.ndarray]
Source:   
def get_matchup_features(
    home_team: str,
    away_team: str,
    final_features: List[str],
    year: Optional[int] = None,
) -> Optional[np.ndarray]:
    """
    Build a feature vector for a matchup in the order of *final_features*.

    Returns None if data is unavailable for either team.
    """
    import sys

    def _log(msg: str) -> None:
        print(f"[serve_features] {msg}", file=sys.stderr)

    year = int(year or datetime.now().year)

    home_abb = TEAM_NAME_TO_ABB.get(home_team)
    away_abb = TEAM_NAME_TO_ABB.get(away_team)
    if home_abb is None:
        _log(f"unknown home team '{home_team}' — not in TEAM_NAME_TO_ABB")
        return None
    if away_abb is None:
        _log(f"unknown away team '{away_team}' — not in TEAM_NAME_TO_ABB")
        return None

    batting, pitching = _get_cac

In [3]:
from model.serve_features import get_matchup_features
get_matchup_features("Boston Red Sox",'New York Yankees',None, 2026)

TypeError: 'NoneType' object is not iterable

In [9]:
predictor._get_features("BOS",'NYY')

In [3]:
predictor.predict??

Signature: predictor.predict(home_team: str, away_team: str) -> dict
Source:   
    def predict(self, home_team: str, away_team: str) -> dict:
        """
        Returns:
          {
            "home_win_prob": float,
            "away_win_prob": float,
            "predicted_winner": str,
            "confidence": str,   # "low" | "medium" | "high"
            "model_used": str,
          }
        """
        if self._pipeline is not None:
            try:
                feats = self._get_features(home_team, away_team)
                if feats is not None:
                    proba = self._pipeline.predict_proba(feats.reshape(1, -1))[0]
                    home_prob = float(proba[1])
                    label = "ML Model v1" if self._final_features else "ML Model v2"
                    return _format_prediction(home_team, away_team, home_prob, label)
                reason = "feature vector returned None"
            except Exception as exc:
                reason = f"{type(exc).

## 2. Probability constraints

In [ ]:
result = predictor.predict('Los Angeles Dodgers', 'San Francisco Giants')

# Probs are between 0 and 1
assert 0.0 <= result['home_win_prob'] <= 1.0, f'home_win_prob out of range: {result["home_win_prob"]}'
assert 0.0 <= result['away_win_prob'] <= 1.0, f'away_win_prob out of range: {result["away_win_prob"]}'

# Probs sum to 1
total = round(result['home_win_prob'] + result['away_win_prob'], 6)
assert total == 1.0, f'Probs do not sum to 1: {total}'

print(f'home_win_prob : {result["home_win_prob"]}')
print(f'away_win_prob : {result["away_win_prob"]}')
print(f'sum           : {total}')
print('Probability constraints: OK')

## 3. Predicted winner matches probability

In [ ]:
matchups = [
    ('New York Yankees', 'Boston Red Sox'),
    ('Los Angeles Dodgers', 'San Francisco Giants'),
    ('Houston Astros', 'Texas Rangers'),
    ('Atlanta Braves', 'New York Mets'),
]

for home, away in matchups:
    r = predictor.predict(home, away)
    expected_winner = home if r['home_win_prob'] >= 0.5 else away
    assert r['predicted_winner'] == expected_winner, (
        f'{home} vs {away}: predicted_winner={r["predicted_winner"]} '
        f'but home_prob={r["home_win_prob"]}'
    )

print('Predicted winner consistency: OK')

## 4. Confidence thresholds

In [ ]:
# Test _format_prediction directly with known probabilities
cases = [
    (0.52, 'low'),
    (0.54, 'low'),
    (0.56, 'medium'),
    (0.64, 'medium'),
    (0.65, 'high'),
    (0.80, 'high'),
    # Symmetric: away favored
    (0.44, 'low'),    # away_prob = 0.56 → medium? No, win_prob = max(0.44, 0.56) = 0.56 → medium
    (0.30, 'high'),   # away_prob = 0.70 → high
]

for prob, expected_conf in cases:
    r = _format_prediction('TeamA', 'TeamB', prob, 'test')
    assert r['confidence'] == expected_conf, (
        f'prob={prob}: expected confidence={expected_conf}, got {r["confidence"]}'
    )
    print(f'  home_prob={prob:.2f} → confidence={r["confidence"]}  ✓')

print('Confidence thresholds: OK')

## 5. Unknown / invalid team names

In [ ]:
# Unknown team should not crash — falls back to baseline
r = predictor.predict('Unknown Team FC', 'New York Yankees')
print('Unknown home team:', r)
assert 'home_win_prob' in r
assert 0.0 <= r['home_win_prob'] <= 1.0

r2 = predictor.predict('New York Yankees', 'Not A Real Team')
print('Unknown away team:', r2)
assert 'home_win_prob' in r2

r3 = predictor.predict('', '')
print('Empty strings    :', r3)
assert 'home_win_prob' in r3

print('Unknown team handling: OK')

## 6. All 30-team matchups

In [ ]:
# Every known full-name team should return a valid prediction
all_teams = [t for t in TEAM_NAME_TO_ABB.keys() if len(t) > 5]  # skip short alias keys

failures = []
rows = []

# Test each team as home vs a fixed away team
away_opponent = 'Boston Red Sox'
for home in all_teams:
    if home == away_opponent:
        continue
    try:
        r = predictor.predict(home, away_opponent)
        assert set(r.keys()) >= {'home_win_prob', 'away_win_prob', 'predicted_winner', 'confidence'}
        assert 0.0 <= r['home_win_prob'] <= 1.0
        rows.append({'home': home, **r})
    except Exception as e:
        failures.append(f'{home}: {e}')

if failures:
    print('FAILURES:')
    for f in failures:
        print(' ', f)
else:
    print(f'All {len(rows)} home teams produced valid predictions.')

pd.DataFrame(rows)[['home', 'home_win_prob', 'predicted_winner', 'confidence', 'model_used']]

## 7. Baseline fallback produces correct value

In [ ]:
# Force baseline by calling _format_prediction directly
r = _format_prediction('TeamA', 'TeamB', 0.54, 'Baseline (home field)')
assert r['home_win_prob'] == 0.54
assert r['away_win_prob'] == 0.46
assert r['model_used'] == 'Baseline (home field)'
assert r['predicted_winner'] == 'TeamA'  # home team favored at 54%
assert r['confidence'] == 'low'          # 54% < 55% threshold

print('Baseline fallback: OK')
print(r)

## 8. Prediction consistency (same input → same output)

In [ ]:
home, away = 'New York Yankees', 'Boston Red Sox'
results = [predictor.predict(home, away) for _ in range(5)]

probs = [r['home_win_prob'] for r in results]
assert len(set(probs)) == 1, f'Non-deterministic predictions: {probs}'

print(f'5 predictions for {home} vs {away}: all {probs[0]}')
print('Consistency: OK')

## 9. Prediction summary

In [ ]:
featured_matchups = [
    ('New York Yankees',     'Boston Red Sox'),
    ('Los Angeles Dodgers',  'San Francisco Giants'),
    ('Houston Astros',       'Texas Rangers'),
    ('Atlanta Braves',       'New York Mets'),
    ('Chicago Cubs',         'Chicago White Sox'),
    ('Philadelphia Phillies','Washington Nationals'),
    ('Minnesota Twins',      'Cleveland Guardians'),
    ('Tampa Bay Rays',       'Baltimore Orioles'),
]

rows = []
for home, away in featured_matchups:
    r = predictor.predict(home, away)
    rows.append({
        'home': home,
        'away': away,
        'home_win_prob': r['home_win_prob'],
        'away_win_prob': r['away_win_prob'],
        'winner': r['predicted_winner'],
        'confidence': r['confidence'],
        'model': r['model_used'],
    })

pd.DataFrame(rows)